# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nandini1313-cloud/flyrank-ml1-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# ============================================================
# 1. RANKED ACTIONS + REASON CODES
# ============================================================

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

# Find the repository root
ROOT = Path.cwd()
while not (ROOT / "outputs" / "refresh_queue.csv").exists():
    ROOT = ROOT.parent

# Load existing validated Week 6 outputs
queue = pd.read_csv(ROOT / "outputs" / "refresh_queue.csv")
model_results = json.loads(
    (ROOT / "outputs" / "model_results.json").read_text()
)

# Add a simple content archetype
def get_archetype(row):
    action = row["suggested_action"]

    if action == "expand_and_refresh":
        return "Thin content"
    elif action == "refresh_and_review_ctr":
        return "CTR opportunity"
    elif action == "refresh_and_review_engagement":
        return "Engagement opportunity"
    elif action == "refresh":
        return "Declining or stale content"
    else:
        return "Monitor"

queue["archetype"] = queue.apply(get_archetype, axis=1)

# Keep the validated ranking
queue = queue.sort_values("final_rank").reset_index(drop=True)

display(
    queue[
        [
            "final_rank",
            "content_id",
            "final_refresh_score",
            "confidence",
            "suggested_action",
            "archetype",
            "final_reason_codes",
        ]
    ].head(20)
)

print("Total content items:", len(queue))
print("Best model:", model_results["best_model"]["name"])
print(
    "Precision@50:",
    round(
        model_results["models"][
            model_results["best_model"]["name"]
        ]["precision_at_50"],
        3,
    ),
)

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# ============================================================
# 2. INTENDED USE + LIMITS
# ============================================================

print("INTENDED USE")
print("- Rank existing content for human review.")
print("- Prioritize pages where a refresh may be useful.")
print("- Use reason codes to understand why an item was ranked.")
print("- Use the score as directional decision support.")

print("\nLIMITS")
print("- The model identifies observed decline-risk patterns.")
print("- It does not prove that refreshing a page will improve results.")
print("- It does not predict Google's algorithm.")
print("- It should not be used as an automatic publishing system.")
print("- Final decisions require human review.")

print("\nMODEL RECEIPT")
best_model = model_results["best_model"]["name"]
print("Best model:", best_model)
print(
    "Precision@50:",
    round(
        model_results["models"][best_model]["precision_at_50"],
        3,
    ),
)
print("Validation:", model_results["split_strategy"])

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# ============================================================
# 3. HUMAN REVIEW + NO-GO LIST
# ============================================================

print("HUMAN REVIEW RULES")
print("1. Open the recommended page.")
print("2. Check whether the reason code matches the page.")
print("3. Check current search intent and user needs.")
print("4. Check whether information is outdated.")
print("5. Check whether the proposed change is useful.")
print("6. Human reviewer makes the final decision.")

print("\nNO-GO AUTOMATION LIST")
print("- Do not automatically publish content.")
print("- Do not automatically delete content.")
print("- Do not automatically rewrite important facts.")
print("- Do not automatically change medical, legal, financial, or safety information.")
print("- Do not automatically send client-facing recommendations.")
print("- Do not treat a model score as proof of causal improvement.")

print("\nARCHETYPE → ACTION")
mapping = pd.DataFrame({
    "Archetype": [
        "Thin content",
        "CTR opportunity",
        "Engagement opportunity",
        "Declining or stale content",
        "Monitor",
    ],
    "Recommended action": [
        "Expand and refresh",
        "Refresh and review CTR",
        "Refresh and review engagement",
        "Refresh",
        "Monitor",
    ],
})

display(mapping)

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# ============================================================
# 4. DECAY / REFRESH + MONITORING + RETRAIN + VALUE
# ============================================================

# Observed decline rate by freshness tier
decay = (
    queue.groupby("freshness_tier")["is_declining_label"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(
        columns={
            "count": "items",
            "mean": "observed_decline_rate",
        }
    )
)

print("DECAY / FRESHNESS INSIGHT")
display(decay)

print(
    "Use freshness as a review signal, not as proof that age causes decline."
)

print("\nMONITORING TRIGGERS")
monitoring = pd.DataFrame({
    "Trigger": [
        "Precision@50 falls materially",
        "Decline rate changes materially",
        "Content mix changes",
        "Feature definitions change",
        "Data quality changes",
    ],
    "Response": [
        "Review ranking quality",
        "Check for population or label shift",
        "Recheck whether actions remain useful",
        "Revalidate the pipeline",
        "Fix data before using the queue",
    ],
})

display(monitoring)

print("\nRETRAIN TRIGGERS")
print("- Retrain only after meaningful data or population change.")
print("- Retrain after sustained ranking degradation.")
print("- Retrain after feature or label-definition changes.")
print("- Re-run holdout validation before accepting a new model.")

print("\nCOST / VALUE")
print("- Start human review with the highest-ranked items.")
print("- Prefer high-visibility pages when reviewer time is limited.")
print("- Prioritize actions with clear reason codes.")
print("- Treat value as prioritization guidance, not measured financial ROI.")

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# ============================================================
# 5. EXPORTS FOR THE PAPER + SELF-CHECK
# ============================================================

output_dir = ROOT / "work" / "outputs"
figure_dir = ROOT / "work" / "figures"

output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

# Paper-ready ranked queue
export_columns = [
    "final_rank",
    "content_id",
    "final_refresh_score",
    "confidence",
    "suggested_action",
    "archetype",
    "final_reason_codes",
    "best_model_probability",
    "is_declining_label",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
]

queue[export_columns].to_csv(
    output_dir / "refresh_action_queue.csv",
    index=False,
)

# Top 50 queue for easy paper review
queue[export_columns].head(50).to_csv(
    output_dir / "refresh_action_queue_top50.csv",
    index=False,
)

# Reusable action figure
action_counts = queue["suggested_action"].value_counts()

plt.figure(figsize=(8, 5))
action_counts.plot(kind="barh")
plt.xlabel("Number of content items")
plt.ylabel("Suggested action")
plt.title("Week 7 Content Action Queue")
plt.tight_layout()
plt.savefig(
    figure_dir / "action_mix_w07.png",
    dpi=180,
)
plt.close()

# Self-check
required_columns = [
    "final_rank",
    "content_id",
    "suggested_action",
    "final_reason_codes",
    "confidence",
]

missing = [
    column for column in required_columns
    if column not in queue.columns
]

print("SELF-CHECK")
print("Rows in queue:", len(queue))
print("Missing required columns:", missing)
print(
    "Queue exported:",
    (output_dir / "refresh_action_queue.csv").exists(),
)
print(
    "Figure exported:",
    (figure_dir / "action_mix_w07.png").exists(),
)

if not missing and len(queue) > 0:
    print("\nWEEK 7 ACTION PLAYBOOK: PASS")
else:
    print("\nWEEK 7 ACTION PLAYBOOK: CHECK REQUIRED")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.